# HPM 数字化电磁算法 CAE V1.3：插件式传播后端快速复现

本笔记本演示：载入全中文工程、查看已注册传播后端、执行混合场景静态控场、对比四类后端，并输出归一化结果。

> 数值边界：仅处理波长尺度、归一化标量复场和无量纲代理指标；不输出绝对源功率、现实器件阈值、毁伤概率或作用距离。

In [ ]:
from pathlib import Path
import sys

项目根目录 = Path.cwd().resolve()
if 项目根目录.name == "notebooks":
    项目根目录 = 项目根目录.parent
sys.path.insert(0, str(项目根目录 / "src"))

from hpm_platform.physics.field_backends import available_field_backends
from hpm_platform.ui.backend_explorer import run_backend_comparison, make_backend_gallery
from hpm_platform.ui.figures import make_field_figure
from hpm_platform.ui.project_model import CAEProject
from hpm_platform.ui.quick_solver import solve_project

## 1. 载入 V1.3 环境工程

In [ ]:
工程 = CAEProject.load_yaml(项目根目录 / "configs" / "cae_project_v13.yaml")
print("工程名称：", 工程.meta.name)
print("传播后端：", 工程.propagation.backend)
print("启用反射面：", len(工程.active_reflectors))
print("启用孔缝：", len(工程.active_apertures))
print("启用腔体：", len(工程.active_cavities))

## 2. 查看传播后端注册表

In [ ]:
for 后端 in available_field_backends():
    print(f"{后端.backend_id:24s}  {后端.display_name}  —  {后端.description}")

## 3. 执行混合场景静态控场

In [ ]:
结果 = solve_project(工程)
结果.metrics_frame()

In [ ]:
make_field_figure(结果)

## 4. 在同一工程约束下对比四类传播后端

In [ ]:
对比 = run_backend_comparison(工程, fast_mode=True)
对比.records

In [ ]:
make_backend_gallery(对比)

## 5. 解读原则

不同后端产生的差异用于验证插件接口、模型敏感性和算法鲁棒性，不意味着某个降阶后端具有普遍的物理优越性。论文中应明确写出每种后端的适用条件、归一化方式和失效边界。